In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from libero.libero import benchmark, get_libero_path
import os
from termcolor import colored

datasets_default_path = get_libero_path("datasets")
bddl_files_default_path = get_libero_path("bddl_files")

print("Default dataset root path: ", datasets_default_path)
print("Default bddl files root path: ", bddl_files_default_path)

benchmark_dict = benchmark.get_benchmark_dict()
print(benchmark_dict)

Default dataset root path:  /home/mila/o/ozgur.aslan/git/LIBERO/libero/libero/../datasets
Default bddl files root path:  /home/mila/o/ozgur.aslan/git/LIBERO/libero/libero/./bddl_files
{'libero_spatial': <class 'libero.libero.benchmark.LIBERO_SPATIAL'>, 'libero_object': <class 'libero.libero.benchmark.LIBERO_OBJECT'>, 'libero_goal': <class 'libero.libero.benchmark.LIBERO_GOAL'>, 'libero_90': <class 'libero.libero.benchmark.LIBERO_90'>, 'libero_10': <class 'libero.libero.benchmark.LIBERO_10'>, 'libero_100': <class 'libero.libero.benchmark.LIBERO_100'>}


In [ ]:
benchmark_name = "libero_goal"
task_id = 0
benchmark_instance = benchmark_dict[benchmark_name]()

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [20]:
import libero.libero.utils.download_utils as download_utils


libero_datasets_exist = download_utils.check_libero_dataset(download_dir=datasets_default_path)

if not libero_datasets_exist:
    download_utils.libero_dataset_download(download_dir=datasets_default_path, datasets="libero_goal")


[ ] Dataset libero_object not found!!!
[ ] Dataset libero_goal not found!!!
[X] Dataset libero_spatial is complete
[ ] Dataset libero_10 not found!!!
[ ] Dataset libero_90 not found!!!
Using original download links (these may expire soon)


iv5e4dos8yy2b212pkzkpxu9wbdgjfeg.zip: 2.88GB [00:33, 85.2MB/s]                                


In [ ]:
# Check if the demo files exist
example_demo_file = os.path.join(datasets_default_path, benchmark_instance.get_task_demonstration(task_id))
if not os.path.exists(example_demo_file):
    print(colored(f"[error] demo file {example_demo_file} cannot be found. Check your paths", "red"))

In [5]:
import h5py
from libero.libero.utils.dataset_utils import get_dataset_info
import imageio

# Print the dataset info. We have a standalone script for doing the same thing available at `scripts/get_dataset_info.py`
get_dataset_info(example_demo_file)

with h5py.File(example_demo_file, "r") as f:
    demo_images = f[f"data/demo_0/obs/agentview_rgb"][()]
    demo_actions = f[f"data/demo_0/actions"][()]
    demo_init_states = f[f"data/demo_0/states"][()]

video_writer = imageio.get_writer("output.mp4", fps=60)
for image in demo_images:
    video_writer.append_data(image[::-1])
video_writer.close()


total transitions: 7027
total trajectories: 50
traj length mean: 140.54
traj length std: 18.237554660644612
traj length min: 116
traj length max: 196
action min: -1.0
action max: 0.9375
language instruction: open the middle drawer of the cabinet

==== Filter Keys ====
no filter keys


==== Env Meta ====
{
    "type": 1,
    "env_name": "Libero_Tabletop_Manipulation",
    "problem_name": "libero_tabletop_manipulation",
    "bddl_file": "chiliocosm/bddl_files/libero_goal/open_the_middle_layer_of_the_drawer.bddl",
    "env_kwargs": {
        "robots": [
            "Panda"
        ],
        "controller_configs": {
            "type": "OSC_POSE",
            "input_max": 1,
            "input_min": -1,
            "output_max": [
                0.05,
                0.05,
                0.05,
                0.5,
                0.5,
                0.5
            ],
            "output_min": [
                -0.05,
                -0.05,
                -0.05,
                -0.5,


In [12]:
from libero.libero.envs import OffScreenRenderEnv, DenseRewardEnv


# task_id is the (task_id + 1)th task in the benchmark
task = benchmark_instance.get_task(task_id)
print("Language description:", task.language)
env_args = {
    "bddl_file_name": os.path.join(bddl_files_default_path, task.problem_folder, task.bddl_file),
    "camera_heights": 1024,
    "camera_widths": 1088
}

env = DenseRewardEnv(**env_args)
# Fix random seeds for reproducibility
if True:
    env.seed(0)
    env.reset()
    env.set_init_state(demo_init_states[0])


    dummy_action = [0.] * 7
    for step in range(20):
        obs, reward, done, info = env.step(dummy_action)

    images = []
    for t, action in enumerate(demo_actions):
        obs, reward, done, info = env.step(action)
        
        print(f"Reward at step {t}: {reward}")
        #img = Image.fromarray()
        images.append(obs["agentview_image"])
        if done:
            break


    video_writer = imageio.get_writer("replay.mp4", fps=60)
    for image in images:
        video_writer.append_data(image[::-1])
    video_writer.close()

Language description: open the middle drawer of the cabinet
Reward at step 0: 0.5634017005364198
Reward at step 1: 0.5649754339069168
Reward at step 2: 0.5674793828010452
Reward at step 3: 0.5708954167974986
Reward at step 4: 0.5747994662759337
Reward at step 5: 0.578731957655387
Reward at step 6: 0.5825544042445717
Reward at step 7: 0.5863419352757885
Reward at step 8: 0.590100911191423
Reward at step 9: 0.5937660048731729
Reward at step 10: 0.5973810908230799
Reward at step 11: 0.6009902788459828
Reward at step 12: 0.6045043977839846
Reward at step 13: 0.6076716782778475
Reward at step 14: 0.6104997127892384
Reward at step 15: 0.6132114780836092
Reward at step 16: 0.6158159042609535
Reward at step 17: 0.6180471281954096
Reward at step 18: 0.6197554490933466
Reward at step 19: 0.6211844890961848
Reward at step 20: 0.6225841709938869
Reward at step 21: 0.6239338627996054
Reward at step 22: 0.6251980407254425
Reward at step 23: 0.6263989761101152
Reward at step 24: 0.6275001391852572
Re

# All In One 

First, generates videos of the trajectories. Then, replays them and prints the reward and save a high quality video.

In [14]:

import os
import h5py
import imageio
from termcolor import colored

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import DenseRewardEnv

datasets_default_path = get_libero_path("datasets")
bddl_files_default_path = get_libero_path("bddl_files")


benchmark_dict = benchmark.get_benchmark_dict()


benchmark_name = "libero_goal"
benchmark_instance = benchmark_dict[benchmark_name]()
for task_id in range(benchmark_instance.n_tasks):
    # Check if the demo files exist
    example_demo_file = os.path.join(datasets_default_path, benchmark_instance.get_task_demonstration(task_id))
    if not os.path.exists(example_demo_file):
        print(colored(f"[error] demo file {example_demo_file} cannot be found. Check your paths", "red"))

    with h5py.File(example_demo_file, "r") as f:
        demo_images = f[f"data/demo_0/obs/agentview_rgb"][()]
        demo_actions = f[f"data/demo_0/actions"][()]
        demo_init_states = f[f"data/demo_0/states"][()]

    video_writer = imageio.get_writer(f"{benchmark_name}_{task_id}_video.mp4", fps=60)
    for image in demo_images:
        video_writer.append_data(image[::-1])
    video_writer.close()


    # task_id is the (task_id + 1)th task in the benchmark
    task = benchmark_instance.get_task(task_id)
    print("Benchmark, task_id: "benchmark_name, task_id, "Language description:", task.language)
    env_args = {
        "bddl_file_name": os.path.join(bddl_files_default_path, task.problem_folder, task.bddl_file),
        "camera_heights": 1024,
        "camera_widths": 1088
    }

    env = DenseRewardEnv(**env_args)
    # Fix random seeds for reproducibility
    if True:
        env.seed(0)
        env.reset()
        env.set_init_state(demo_init_states[0])


        dummy_action = [0.] * 7
        for step in range(20):
            obs, reward, done, info = env.step(dummy_action)

        images = []
        for t, action in enumerate(demo_actions):
            obs, reward, done, info = env.step(action)
            
            print(f"Reward at step {t}: {reward}")
            #img = Image.fromarray()
            images.append(obs["agentview_image"])
            if done:
                break


        video_writer = imageio.get_writer(f"{benchmark_name}_{task_id}_replay.mp4", fps=60)
        for image in images:
            video_writer.append_data(image[::-1])
        video_writer.close()

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Language description: open the middle drawer of the cabinet
Reward at step 0: 0.5634017005364198
Reward at step 1: 0.5649754339069168
Reward at step 2: 0.5674793828010452
Reward at step 3: 0.5708954167974986
Reward at step 4: 0.5747994662759337
Reward at step 5: 0.578731957655387
Reward at step 6: 0.5825544042445717
Reward at step 7: 0.5863419352757885
Reward at step 8: 0.590100911191423
Reward at step 9: 0.5937660048731729
Reward at step 10: 0.5973810908230799
Reward at step 11: 0.6009902788459828
Reward at step 12: 0.6045043977839846
Reward at step 13: 0.6076716782778475
Reward at step 14: 0.6104997127892384
Reward at step 15: 0.6132114780836092
Reward at step 16: 0.6158159042609535
Reward at step 17: 0.6180471281954096
Reward at step 18: 0.6197554490933466
Reward at step 19: 0.6211844890961848
Reward at step 20: 0.6225841709938869
Reward at step 21: 0.6239338627996054
Reward at step 22: 0.6251980407254425
Reward at step 23: 0.6